# Depth Point Refinement — Nordland

Builds the **refined filled DEM depth** raster (`dem_depth`) for Nordland
using DEM + sea depth polygons + Dybdepunkt data and saves as GeoTIFF.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio as rio

import mnk.substrat as subkart
import mnk.sources

REGION_LIST = ["Nordland"]
RES = 50
CRS = "EPSG:25833"
OUT_DIR = Path("features")
OUT_DIR.mkdir(exist_ok=True)

## 1. Load data sources

In [ ]:
gdf_sea_map = mnk.sources.sea_map_basisdata(REGION_LIST)
gdf_sea_map = subkart.features.depth_preprocess(gdf_sea_map, is_rerun=True)
transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map, res=RES)
print(f"Sea map polygons: {len(gdf_sea_map)}, raster: {out_shape}")

In [ ]:
dem_norge = mnk.sources.dem_data()
dem = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, CRS)
del dem_norge

In [ ]:
POINT_PARQUET = Path("../geonorge/Basisdata_18_Nordland_25833_Dybdepunkt.geo.parquet")
if not POINT_PARQUET.exists():
    print("Preparing point data from GML (this may take a few minutes)...")
    gdf_points = gpd.read_file(
        "../geonorge/Basisdata_18_Nordland_25833_Dybdedata_GML.gml",
        layer="Dybdepunkt", columns=["dybde"],
    )[["dybde", "geometry"]]
    gdf_points.to_parquet(POINT_PARQUET, compression="snappy")
else:
    gdf_points = gpd.read_parquet(POINT_PARQUET)

print(f"Depth points: {len(gdf_points):,}")

## 2. Build refined filled DEM depth

In [ ]:
%%time
interp_depth = subkart.features.interpolate_depth_raster(
    gdf_sea_map, bounds, RES, np.float32, gdf_points=gdf_points,
)

depth = np.ma.filled(dem.data, np.nan).astype(np.float32, copy=False)
dem_depth = np.where(np.isnan(depth), -interp_depth, depth).astype(np.float32)
print(f"dem_depth shape: {dem_depth.shape}")

## 3. Save as GeoTIFF

In [ ]:
nodata = -9999.0
band = np.where(np.isnan(dem_depth), nodata, dem_depth)
out_path = OUT_DIR / "nordland_dem_depth_points.tif"

with rio.open(
    out_path, "w", driver="GTiff",
    height=out_shape[0], width=out_shape[1], count=1,
    dtype=np.float32, crs=CRS, transform=transform, nodata=nodata,
) as dst:
    dst.write(band, 1)

print(f"Saved {out_path}")